# CUDA GEMM Optimization: Phase-by-Phase Implementation

**Target GPU:** NVIDIA T4 (Colab default) — Compute Capability 7.5
**Problem size:** M=N=K=4096

## Setup

In [ ]:
import subprocess, sys, os, glob, shutil, time, textwrap
!nvcc --version
!nvidia-smi
!cat /proc/driver/nvidia/gpus/0/information 2>/dev/null || echo "No GPU info file"

In [ ]:
from google.colab import files
import zipfile, io
print("Upload the cuda-gemm project zip file:")
uploaded = files.upload()
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(uploaded[filename]), 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print(f"Extracted {filename}")
!ls -la /content/cuda-gemm/ 2>/dev/null || !ls -la /content/GEMM/ 2>/dev/null
if os.path.isdir('/content/cuda-gemm'):
    os.chdir('/content/cuda-gemm')
elif os.path.isdir('/content/GEMM'):
    os.chdir('/content/GEMM')
print(f"Working directory: {os.getcwd()}")

In [ ]:
def compile_and_run(kernel_name, extra_flags=""):
    import subprocess
    src = f"kernels/{kernel_name}.cu"
    exe = f"kernels/{kernel_name}"
    arch = "-arch=sm_75"
    cmd = f"nvcc {arch} -std=c++17 -O3 {extra_flags} -o {exe} {src} -lcublas -lcudart"
    print(f"Compiling: {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print("COMPILATION FAILED")
        print("=== stdout ===")
        print(p.stdout[-2000:] if len(p.stdout) > 2000 else p.stdout)
        print("=== stderr ===")
        print(p.stderr[-2000:] if len(p.stderr) > 2000 else p.stderr)
        return False
    print(f"Running {kernel_name}...")
    p = subprocess.run(f"./{exe}", shell=True, capture_output=True, text=True)
    print("=== stdout ===")
    print(p.stdout[-5000:] if len(p.stdout) > 5000 else p.stdout)
    if p.stderr:
        print("=== stderr ===")
        print(p.stderr[-2000:] if len(p.stderr) > 2000 else p.stderr)
    if p.returncode != 0:
        print(f"FAILED (exit code {p.returncode})")
        return False
    print("PASS")
    return True

results = {}

---
## Phase 0: Naive Kernel (Baseline)

In [ ]:
print("=" * 60)
print("Phase 0: Naive Kernel")
print("=" * 60)
ok = compile_and_run("00_naive")
results["00_naive"] = ok

---
## Phase 2: Shared-Memory Tiling

In [ ]:
print("=" * 60)
print("Phase 2: Shared-Memory Tiling")
print("=" * 60)
ok = compile_and_run("02_shared_tiled")
results["02_shared_tiled"] = ok

---
## Phase 3: 1D Register Blocking

In [ ]:
print("=" * 60)
print("Phase 3: 1D Register Blocking")
print("=" * 60)
ok = compile_and_run("03_1d_blocktiling")
results["03_1d_blocktiling"] = ok

---
## Phase 4: 2D Register Blocking

In [ ]:
print("=" * 60)
print("Phase 4: 2D Register Blocking")
print("=" * 60)
ok = compile_and_run("04_2d_blocktiling")
results["04_2d_blocktiling"] = ok

---
## Phase 5: Vectorized Memory Access (float4)

In [ ]:
print("=" * 60)
print("Phase 5: Vectorized Memory Access")
print("=" * 60)
ok = compile_and_run("05_vectorized")
results["05_vectorized"] = ok

---
## Phase 6: Bank-Conflict-Free Shared Memory

In [ ]:
print("=" * 60)
print("Phase 6: Bank-Conflict-Free")
print("=" * 60)
ok = compile_and_run("06_bank_conflict_free")
results["06_bank_conflict_free"] = ok

---
## Phase 7: Double Buffering

In [ ]:
print("=" * 60)
print("Phase 7: Double Buffering")
print("=" * 60)
ok = compile_and_run("07_double_buffered")
results["07_double_buffered"] = ok

---
## Phase 8: Fully Vectorized (float4 A+B+C)

In [ ]:
print("=" * 60)
print("Phase 8: Fully Vectorized (A+B+C)")
print("=" * 60)
ok = compile_and_run("08_vectorized_all")
results["08_vectorized_all"] = ok

---
## Phase 9: Tensor Core via WMMA

In [ ]:
print("=" * 60)
print("Phase 9: Tensor Core WMMA")
print("=" * 60)
ok = compile_and_run("09_wmma_tensorcore")
results["09_wmma_tensorcore"] = ok

---
## Phase 10: Multi-Tile WMMA (2x2 per warp)

In [ ]:
print("=" * 60)
print("Phase 10: Multi-Tile WMMA")
print("=" * 60)
ok = compile_and_run("10_wmma_multi")
results["10_wmma_multi"] = ok

---
## Phase 11: Direct-Load WMMA

In [ ]:
print("=" * 60)
print("Phase 11: Direct-Load WMMA")
print("=" * 60)
ok = compile_and_run("11_wmma_direct")
results["11_wmma_direct"] = ok

---
## Phase 12: Optimized WMMA (BM=128, 512 threads)

In [ ]:
print("=" * 60)
print("Phase 12: Optimized WMMA")
print("=" * 60)
ok = compile_and_run("12_wmma_opt")
results["12_wmma_opt"] = ok

---
## Phase 13: Combined WMMA (direct B load)

In [ ]:
print("=" * 60)
print("Phase 13: Combined WMMA")
print("=" * 60)
ok = compile_and_run("13_wmma_combined")
results["13_wmma_combined"] = ok

---
## Phase 14: Bank-Conflict-Free Padded WMMA

Adds BK+1 padding to As/Bs_trans and BN+1 to Bs_raw to eliminate bank conflicts on transpose writes and fragment loads.

In [ ]:
print("=" * 60)
print("Phase 14: Padded (Conflict-Free) WMMA")
print("=" * 60)
ok = compile_and_run("14_wmma_padded")
results["14_wmma_padded"] = ok

---
## Phase 15: Double-Buffered (Pipelined) WMMA

2-stage software pipeline: prefetches next tile into registers while computing the current tile. Register staging overlaps global memory latency with tensor-core compute.

In [ ]:
print("=" * 60)
print("Phase 15: Pipelined WMMA")
print("=" * 60)
ok = compile_and_run("15_wmma_pipelined")
results["15_wmma_pipelined"] = ok

---
## Phase 16: Pipelined WMMA — No B Transpose

Eliminates the shared-memory B transpose entirely. B in global memory is already row-major K×N, which is the native layout for a `row_major` matrix_b fragment. Removing the transpose drops one shared array, one scalar-copy pass (BK×BN elements/tile), and one `__syncthreads()` per iteration.

In [ ]:
print("=" * 60)
print("Phase 16: No-Transpose WMMA")
print("=" * 60)
ok = compile_and_run("16_wmma_notranspose")
results["16_wmma_notranspose"] = ok

---
## Phase 17: Deeper K-Tile (BK=32) WMMA

Doubles BK from 16 to 32 with an inner WMMA_K sub-loop. Each shared-memory tile feeds two tensor-core passes (kk=0, 16) before fetching the next tile. Halves outer-loop iterations and sync overhead.

In [ ]:
print("=" * 60)
print("Phase 17: Deep-K (BK=32) WMMA")
print("=" * 60)
ok = compile_and_run("17_wmma_deepk")
results["17_wmma_deepk"] = ok

---
## Phase 18: Deep-K WMMA + Threadblock Swizzle

Adds a CUTLASS-style threadblock swizzle (`GROUP_M=8`) to reorder which output tile each block computes. Consecutively-launched blocks share B column-tile ranges for better L2 reuse. Same BK=32, no-B-transpose structure as Phase 17.

In [ ]:
print("=" * 60)
print("Phase 18: Swizzled WMMA")
print("=" * 60)
ok = compile_and_run("18_wmma_swizzled")
results["18_wmma_swizzled"] = ok

In [ ]:
!nvcc -arch=sm_75 -std=c++17 -O3 -Xptxas -v -c kernels/17_wmma_deepk.cu -o /dev/null

---
## Combined Benchmark

In [ ]:
print("=" * 60)
print("Combined Benchmark: All Kernels vs cuBLAS")
print("=" * 60)

arch = "-arch=sm_75"
kernel_files = " ".join([
    "kernels/00_naive.cu",
    "kernels/02_shared_tiled.cu",
    "kernels/03_1d_blocktiling.cu",
    "kernels/04_2d_blocktiling.cu",
    "kernels/05_vectorized.cu",
    "kernels/06_bank_conflict_free.cu",
    "kernels/07_double_buffered.cu",
    "kernels/08_vectorized_all.cu",
    "kernels/09_wmma_tensorcore.cu",
    "kernels/10_wmma_multi.cu",
    "kernels/11_wmma_direct.cu",
    "kernels/12_wmma_opt.cu",
    "kernels/13_wmma_combined.cu",
    "kernels/14_wmma_padded.cu",
    "kernels/15_wmma_pipelined.cu",
    "kernels/16_wmma_notranspose.cu",
    "kernels/17_wmma_deepk.cu",
    "kernels/18_wmma_swizzled.cu",
])

cmd = f"nvcc {arch} -std=c++17 -O3 -DBENCHMARK_ALL -o bench/benchmark_all bench/benchmark_all.cu {kernel_files} -lcublas -lcudart"
print(f"Compiling: nvcc {arch} -std=c++17 -O3 -DBENCHMARK_ALL -o bench/benchmark_all ...")
p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if p.returncode != 0:
    print("COMPILATION FAILED")
    if p.stderr:
        print("=== stderr ===")
        print(p.stderr[-2000:])
else:
    print("Running combined benchmark...")
    p = subprocess.run("./bench/benchmark_all", shell=True, capture_output=True, text=True)
    print(p.stdout)
    if p.stderr:
        print("=== stderr ===")
        print(p.stderr[-2000:] if len(p.stderr) > 2000 else p.stderr)
    if p.returncode != 0:
        print(f"Benchmark FAILED (exit code {p.returncode})")

---
## Optional: Autotuning

In [ ]:
print("Running autotuner (this may take a while)...")
!python3 autotune.py

---
## Summary

In [ ]:
print("Phase Results:")
print("=" * 40)
for name, ok in results.items():
    status = "PASS" if ok else "FAIL"
    print(f"  {name:30s}: {status}")
print("=" * 40)
print("\nBenchmark table (fill from combined benchmark output):")
print('''
| Level | Technique | GFLOPS | % of cuBLAS |
|-------|-----------|--------|-------------|
| 0 | Naive | | |
| 2 | Shared memory tiling | | |
| 3 | 1D register blocking | | |
| 4 | 2D register blocking | | |
| 5 | Vectorized | | |
| 6 | Bank-conflict-free | | |
| 7 | Double buffering | | |
| 8 | Fully vectorized (A+B+C) | | |
| 9 | Tensor core (WMMA) | | |
| 10 | Multi-tile WMMA | | |
| 11 | Direct-load WMMA | | |
| 12 | Optimized WMMA | | |
| 13 | Combined WMMA | | |
| 14 | Padded WMMA | | |
| 15 | Pipelined WMMA | | |
| 16 | No-transpose WMMA | | |
| 17 | Deep-K (BK=32) WMMA | | |
| 18 | Swizzled WMMA | | |
| — | cuBLAS (reference) | | |
''')